# 01 · Modelo 2D básico: voladizo

> **validación humana requerida** antes de usar resultados en ingeniería.

Ejecutamos el fixture propio `cantilever-linear-static.json` mediante Rust.
`linearStatic` figura como `Validated` en el registro local
según el registro local.
`schemas/engine-input/1.json` (línea 5) y `methods.rs` declaran Experimental solo
`nonlinearStatic`, `soilProfile` y `consolidation1d`; pDelta es Stable; ENG-BENCH-002;
Validated pendiente de ENG-VAL-006. Otras capacidades: validación DESCONOCIDO. Este ejercicio comprueba solo la flecha de este caso sintético.
La disponibilidad de un elemento no acredita por sí sola madurez independiente.

Requiere la wheel local instalada; no hay paquete en PyPI (publicación PENDIENTE,
sin decidir). El kernel debe arrancar en la raíz del repo o un subdirectorio suyo.
No requiere NumPy ni pandas. Consulta la [guía de usuario](../docs/GUIA_USUARIO.md).

In [ ]:
from pathlib import Path
import math
import mamut_engine as me

relative = Path("crates/engine-api/fixtures/input/valid/cantilever-linear-static.json")
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents)
             if (p / relative).is_file() and (p / "bindings/python").is_dir()), None)
if root is None:
    raise RuntimeError("Ejecuta dentro del repositorio para localizar el fixture")
model = me.Model.from_json((root / relative).read_text(encoding="utf-8"))
print("Versión y contratos:", me.version())
descriptors = {d["workflow"]["method_id"]: d for d in me.capabilities()["method_descriptors"]}
status = descriptors.get("linearStatic", {}).get("status", "DESCONOCIDO")
print("linearStatic:", "experimental: " + status if status == "Experimental" else status)

## Parámetros y convención

SI y Z-up, D2 en el plano XY (Z=0): la carga actúa en −Y. La punta se identifica
por el ID del extremo j del miembro. E se expresa en Pa, I en m⁴ y L en m.
El apoyo del extremo i está empotrado en Ux, Uy, Rz.

In [ ]:
# Leer geometría, carga y propiedades del documento, sin constantes físicas nuevas.
geometry = model.data["model"]
member = geometry["members"][0]
base_id, tip_id = member["nodes"]
base_node = next(n for n in geometry["nodes"] if n["id"] == base_id)
tip_node = next(n for n in geometry["nodes"] if n["id"] == tip_id)
assert geometry["axes"] == "Z-up" and model.data["units"] == "SI"
assert base_node["dim"] == tip_node["dim"] == "D2"
assert base_node["coordinates"][1:] == tip_node["coordinates"][1:] == [0, 0]
assert tip_node["coordinates"][0] > base_node["coordinates"][0]
L = math.dist(base_node["coordinates"], tip_node["coordinates"])
section = next(s for s in geometry["sections"] if s["id"] == member["section"])
law = next(law for law in geometry["laws"] if law["name"] == member["law"])
I = next(p["value"] for p in section["properties"] if p["name"] == "I")
E = next(p["value"] for p in law["parameters"] if p["name"] == "E")
input_case = model.data["cases"][0]
load_index, tip_load = next((i, load) for i, load in enumerate(input_case["loads"])
                            if load["node"] == tip_id)
component_index, component = next((i, v) for i, v in enumerate(tip_load["values"])
                                if v["dof"] == "Uy")
P = component["value"]  # Componente con signo, N.
assert L > 0 and E > 0 and I > 0 and P != 0
print({"P_N": P, "L_m": L, "E_Pa": E, "I_m4": I})

In [ ]:
support = next(s for s in geometry["supports"] if s["node"] == base_id)
assert set(support["restrained"]) == {"Ux", "Uy", "Rz"}

In [ ]:
report = me.validate(model, lang="es")
print("Diagnósticos de validación:", report["diagnostics"])
assert not any(d["severity"] == "error" for d in report["diagnostics"])
results = me.run(model, lang="es")
assert results.status in ("ok", "warning") and not results.partial
print("Estado:", results.status, "Diagnósticos:", results.diagnostics)
document = results.to_dict()

## Flecha de punta y comprobación analítica

Para flexión lineal Euler–Bernoulli, integrar la curvatura con desplazamiento
y giro nulos en el empotramiento da `Uy(L) = P·L³/(3·E·I)`, con P con signo.
Esta evaluación es una expectativa analítica del test; el análisis lo calcula Rust.
Los vectores nodales usan `[Ux, Uy, Uz, Rx, Ry, Rz]`, por lo que Uy ocupa 1.

Usamos `rel_tol=1e-9` y `abs_tol=0`: la flecha esperada no es cero y el margen
cubre redondeo binary64 de este sistema pequeño. No es una tolerancia general
del motor ni un criterio de aceptación de proyectos. Los parámetros sintéticos
no verifican grandes desplazamientos ni representan un dimensionamiento real.

In [ ]:
case = next(c for c in document["cases"] if c["case"] == input_case["name"])
tip_result = next(n for n in case["nodes"] if n["id"] == tip_id)
uy = tip_result["displacements"][1]
expected_uy = P * L**3 / (3 * E * I)
assert math.isclose(uy, expected_uy, rel_tol=1e-9, abs_tol=0.0), (uy, expected_uy)
print({"Uy_Rust_m": uy, "Uy_analitico_m": expected_uy,
       "error_relativo": abs((uy - expected_uy) / expected_uy)})

In [ ]:
print("Productor y contratos:", results.version)
print("Huella del input:", results.input_sha256)
print("Métodos:", document["methods"], "Target:", document.get("target", "DESCONOCIDO"))
print("Commit:", document.get("git_commit") or "DESCONOCIDO")
assert results.input_sha256 == model.input_sha256